In [1]:
import pandas as pd
import numpy as np
import os
os.environ['CUDA_VISIBLE_DEVICES']='3'

In [2]:
df1 = pd.read_csv('NQ_test.csv')
df1 = df1[['combined_positive','answers','combined_negative']]
df1 = df1.sample(250)
df1['context'] = df1['combined_positive'] + ' ' + df1['combined_negative']
df1 = df1[['context','answers','combined_positive']]
df1['Response'] = df1['answers']
df1.rename(columns={'answers': 'answer', 'combined_positive':'supporting_context'}, inplace=True)
df1.head()

,context,answer,supporting_context,Response
343,(and their first in nine years) was a triumph ...,Ivan Cleary,(and their first in nine years) was a triumph ...,Ivan Cleary
276,Pulmonary artery A pulmonary artery is an arte...,the lungs,Pulmonary artery A pulmonary artery is an arte...,the lungs
445,Raiders of the Lost Ark Raiders of the Lost Ar...,Steven Spielberg,Raiders of the Lost Ark Raiders of the Lost Ar...,Steven Spielberg
258,Bishop of Edmonton in the Anglican Church of C...,2013,Bishop of Edmonton in the Anglican Church of C...,2013
125,"season, consisting of nine episodes, was relea...","October 27 , 2017","season, consisting of nine episodes, was relea...","October 27 , 2017"


In [3]:
df2 = pd.read_csv('test_data_wiki.csv')
df2 = df2[['answer','supporting_context','context']]
df2 = df2.sample(250)
df2['Response'] = df2['answer']
df2.head()

,answer,supporting_context,context,Response
74,University of Buenos Aires,The Escuela Superior Latinoamericana de Inform...,Sue Fennessy( born 25 July 1968) is an Austral...,University of Buenos Aires
237,Song Of Nevada,"Jasper Joseph Inman Kane (March 19, 1894, San ...","Jasper Joseph Inman Kane (March 19, 1894, San ...",Song Of Nevada
421,English,William Neville (15 July 1497 – c. 1545) of Pe...,Abbas Amanat is William Graham Sumner Professo...,English
284,New York,"Leigh Jason( July 26, 1904 – February 19, 1979...","Leigh Jason( July 26, 1904 – February 19, 1979...",New York
16,Tripoli,Vice-Admiral Sir George Tryon (4 January 1832 ...,""" Where Was I?"" may refer to: Beaulieu- sur- L...",Tripoli


In [4]:
df3 = pd.read_csv('LFQA.csv')
df3.head()

,answer,Response
0,"This is tricky to answer. In all probability, ...","This is tricky to answer. In all probability, ..."
1,"First off, it isn't that Sharapova started usi...","First off, it isn't that Sharapova started usi..."
2,/r/fitness has a really great [getting started...,/r/fitness has a really great [getting started...
3,"People die. \n\nSeriously though, the cars all...","People die. \n\nSeriously though, the cars all..."
4,"Surface tension isn't what kills you, it's the...","Surface tension isn't what kills you, it's the..."


In [5]:
df4 = pd.read_csv('HotpotQA_testing_dataset_preprocessed.csv')
df4 = df4[['answer','preprocessed_context']]
df4 = df4.sample(250)
df4['Response']=df4['answer']
df4.rename(columns={'preprocessed_context': 'context'}, inplace=True)
df4.head()

,answer,context,Response
107,Saint Motel,saintmotelevision saintmotelevision stylized a...,Saint Motel
44,1992,time to say goodbye antique song time to say g...,1992
76,15 November 1967,gus poyet gustavo augusto gus poyet domnguez b...,15 November 1967
445,Christian,martyrs memorial patna martyrs memorial also k...,Christian
419,For Love Alone,year of spectacular men year of spectacular me...,For Love Alone


In [6]:
final_df = pd.concat([df1, df2, df3, df4], ignore_index=True)
final_df.head()

,context,answer,supporting_context,Response
0,(and their first in nine years) was a triumph ...,Ivan Cleary,(and their first in nine years) was a triumph ...,Ivan Cleary
1,Pulmonary artery A pulmonary artery is an arte...,the lungs,Pulmonary artery A pulmonary artery is an arte...,the lungs
2,Raiders of the Lost Ark Raiders of the Lost Ar...,Steven Spielberg,Raiders of the Lost Ark Raiders of the Lost Ar...,Steven Spielberg
3,Bishop of Edmonton in the Anglican Church of C...,2013,Bishop of Edmonton in the Anglican Church of C...,2013
4,"season, consisting of nine episodes, was relea...","October 27 , 2017","season, consisting of nine episodes, was relea...","October 27 , 2017"


In [7]:
final_df.shape

(1000, 4)

In [8]:
final_df.to_csv('Testing_dataset.csv',index=False)

In [9]:
import tiktoken
import ast
# Function to count tokens in a string using tiktoken
def count_tokens(text):
    encoding = tiktoken.get_encoding("cl100k_base")
    return len(encoding.encode(text))

# Function to safely parse context - handles both string and list formats
def parse_context(context):
    """
    Parse context whether it's stored as string representation of list or actual list
    """
    if isinstance(context, str):
        try:
            # Try to parse as literal (for string representations of lists)
            parsed = ast.literal_eval(context)
            return parsed
        except (ValueError, SyntaxError):
            # If it's just a plain string, return as is
            return context
    else:
        # If it's already a list or other structure, return as is
        return context

# Function to count tokens for nested context structure
def count_context_tokens(context):
    """
    Count tokens in context, handling different possible formats
    """
    total_tokens = 0
    
    # First parse the context
    parsed_context = parse_context(context)
    
    # Handle different formats
    if isinstance(parsed_context, list):
        for item in parsed_context:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                # Expected format: [title, text_snippets]
                title = item[0]
                text_snippets = item[1]
                
                # Count tokens in title
                if isinstance(title, str):
                    total_tokens += count_tokens(title)
                
                # Count tokens in text snippets
                if isinstance(text_snippets, list):
                    for snippet in text_snippets:
                        if isinstance(snippet, str):
                            total_tokens += count_tokens(snippet)
                elif isinstance(text_snippets, str):
                    total_tokens += count_tokens(text_snippets)
            else:
                # Handle single items or unexpected formats
                if isinstance(item, str):
                    total_tokens += count_tokens(item)
    elif isinstance(parsed_context, str):
        # If it's just a string, count tokens directly
        total_tokens += count_tokens(parsed_context)
    
    return total_tokens


# Apply the count_context_tokens function to the 'context' column
final_df['context_token_count'] = final_df['answer'].apply(count_context_tokens)

In [17]:
final_df.to_csv('Testing_dataset.csv',index=False)